# FiGS — capture to simulated flight

Runs the full pipeline on `intellisense05` end to end: raw phone video, through
Structure-from-Motion and Gaussian-Splat training, to a rendered drone flight.

**Run cells in order.** Every step is idempotent — completed work is detected and skipped, so a
failure part-way through is resumed by re-running from the top rather than starting over. That
matters because two stages are expensive (SuperGlue matching ~38 min, splat training ~34 min).

Each step records into a global `RESULTS` dict, and the final cell writes a run record to
`runs/<scene>_<timestamp>.json` so every execution is self-documenting.

**Before you start:** the kernel must be the `kitchen` env. If you launched Jupyter elsewhere,
stop and relaunch after `source ~/FYP-Radiance/figs/figs_env.sh`. Cell 1 refuses to continue
otherwise, which is deliberate — a wrong kernel produces confusing failures three phases later.

*Validated 2026-08-06 on RTX 2080 8 GB. Reference numbers throughout are from that run.*

In [2]:
# ── 0. Parameters ────────────────────────────────────────────────────────────────
# The only cell you normally edit. Everything downstream derives from these.
from pathlib import Path
import numpy as np

HOME         = Path.home()
PROJECT_ROOT = HOME / "FYP-Radiance" / "figs"          # NOT ~/projects/figs_validation
REPO         = PROJECT_ROOT / "SousVide"

SCENE        = "intellisense_02"                           # scene + capture-config + workspace name
RAW_VIDEO    = HOME / "FYP-Radiance" / "video_captures" / "intellisense_capture_2.mp4"

# ArUco — marker_length is METRES, tape-measured. Nothing downstream can check it.
MARKER_ID     = 0
MARKER_LENGTH = 0.18

# Frame extraction
NUM_IMAGES = 300      # total training frames
NUM_MARKED = 40       # of those, how many must show the marker (raised from the default 20
                      # because our marker detections cluster in time — see Phase 1)

# Transcode target. hloc resizes to 1024 px internally, and nerfstudio auto-downscales to
# <=1600 px, so 4K input is discarded twice over. 1080p30 matches the validated config.
TARGET_W, TARGET_H, TARGET_FPS = 1920, 1080, 30

# Flight
COURSE = "intellisense_loop"
FRAME  = "carl"          # drone physical specs
PILOT  = "Viper"         # MPC tuning
METHOD = "eval_single"   # rollout config
OUT_MP4 = f"{SCENE}_flight.mp4"

# Escape hatches — set True to force a stage to redo work it thinks is already done.
FORCE_TRANSCODE = True
FORCE_GSPLAT    = True

CAPTURE_MP4 = REPO / "gsplats" / "capture" / f"{SCENE}.mp4"
WORKSPACE   = REPO / "gsplats" / "workspace" / SCENE

# Accumulator for the run record. Re-running THIS cell must not wipe results already
# gathered by later cells - but switching SCENE must start clean.
try:
    RESULTS
    if RESULTS.get("scene") != SCENE:
        RESULTS = {"scene": SCENE}
        print("(new scene - RESULTS reset)")
except NameError:
    RESULTS = {"scene": SCENE}
print(f"scene       {SCENE}")
print(f"repo        {REPO}")
print(f"raw video   {RAW_VIDEO}")
print(f"marker      id={MARKER_ID}  length={MARKER_LENGTH} m")

scene       intellisense_02
repo        /home/intellisense05/FYP-Radiance/figs/SousVide
raw video   /home/intellisense05/FYP-Radiance/video_captures/intellisense_capture_2.mp4
marker      id=0  length=0.18 m


In [3]:
# ── 1. Helpers ───────────────────────────────────────────────────────────────────
import subprocess, sys, os, json, time, threading, shutil, re, textwrap
from datetime import datetime

class StepFailed(Exception):
    """Raised with an actionable message rather than a bare traceback."""

def section(title):
    print("\n" + "=" * 78 + f"\n  {title}\n" + "=" * 78)

def ok(msg):   print(f"  ✔ {msg}")
def warn(msg): print(f"  ! {msg}")
def info(msg): print(f"    {msg}")

def sh(cmd, cwd=None, stream=True, check=True):
    """Run a shell command, optionally streaming output. Returns (rc, combined_output)."""
    p = subprocess.Popen(cmd, cwd=cwd, shell=isinstance(cmd, str),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    buf = []
    for line in p.stdout:
        buf.append(line)
        if stream:
            print(line, end="")
    rc = p.wait()
    out = "".join(buf)
    if check and rc != 0:
        raise StepFailed(diagnose(out) or f"command failed (rc={rc}): {cmd}")
    return rc, out

# ── Known failure modes, newest first. Each entry: (regex, explanation) ──────────
_DIAGNOSES = [
    (r"image_list|incompatible function arguments.*import_images",
     "pycolmap/hloc API drift. The vendored hloc calls import_images(image_list=...) but "
     "newer pycolmap renamed it to image_names. Re-run the hloc compatibility cell (Phase 2)."),
    (r"Mismatched number of aruco and sfm transforms",
     "ArUco alignment failed: fewer valid marker detections than NUM_MARKED. Either MARKER_ID "
     "is wrong, or the marker is too sparsely visible. Re-check the Phase 1 scan and consider "
     "lowering NUM_MARKED (at the cost of alignment quality)."),
    (r"CUDA out of memory|CUDA error: out of memory",
     "GPU OOM. Close ns-viewer if running (it holds ~4.6 GB), then retry. If it still OOMs "
     "during training, run ns-train manually with --downscale-factor 2 reusing the existing "
     "transforms.json / sparse_pc.ply."),
    (r"No such file or directory.*gsplats/capture|FileNotFoundError.*capture",
     "The video is not staged in gsplats/capture/. Run the transcode cell (Phase 1)."),
    (r"ValueError.*match|multiple.*match",
     "Ambiguous filename glob in gsplats/capture/ - more than one file contains the scene "
     "name. Keep exactly one unambiguous file per scene."),
    (r"Could not reconstruct any model",
     "COLMAP found no consistent reconstruction. Almost always insufficient overlap or motion "
     "blur during filming. Re-shoot: walk slower, keep generous overlap between views."),
    (r"ModuleNotFoundError: No module named 'sousvide'",
     "Expected and harmless. sousvide is the SOUS VIDE policy-distillation layer, downstream "
     "of everything FiGS does. Not needed for capture, training, or simulation."),
    (r"FATAL: torch changed",
     "STOP. A dependency moved torch off 2.1.2, which silently invalidates tiny-cuda-nn and "
     "gsplat. Rebuild: install_figs.sh --prefix ~/FYP-Radiance/figs "
     "--redo conda_env --redo tcnn --redo pips"),
]

def diagnose(text):
    for pattern, explanation in _DIAGNOSES:
        if re.search(pattern, text, re.I):
            return f"{explanation}\n\n--- matched in output ---\n{text[-1500:]}"
    return None

# ── VRAM sampling ────────────────────────────────────────────────────────────────
def _vram_now():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5).stdout.strip().splitlines()[0]
        return int(out)
    except Exception:
        return -1

class VramMonitor:
    """Samples VRAM in a background thread. Use as a context manager; read .peak after."""
    def __init__(self, interval=5.0):
        self.interval, self.peak, self.baseline = interval, 0, _vram_now()
        self._stop = threading.Event()
    def _loop(self):
        while not self._stop.wait(self.interval):
            self.peak = max(self.peak, _vram_now())
    def __enter__(self):
        self.peak = self.baseline
        self._t = threading.Thread(target=self._loop, daemon=True); self._t.start()
        return self
    def __exit__(self, *a):
        self._stop.set(); self._t.join(timeout=self.interval + 1)
        print(f"\n  VRAM baseline {self.baseline} MiB → peak {self.peak} MiB")

def elapsed(t0):
    s = time.time() - t0
    return f"{int(s//60)}m {s%60:.0f}s" if s >= 60 else f"{s:.1f}s"

ok("helpers loaded")

  ✔ helpers loaded


## Phase 0 — Preflight

Fails fast and loudly rather than letting a wrong environment surface as a confusing error two
hours in. The `kitchen` check is the important one: a Jupyter kernel started outside the env
will import a different torch and everything downstream misbehaves.

In [4]:
# ── 2. Preflight ─────────────────────────────────────────────────────────────────
section("Preflight")

problems = []

# --- environment ---
env = os.environ.get("CONDA_DEFAULT_ENV", "")
if env != "kitchen":
    problems.append(f"conda env is '{env or 'none'}', expected 'kitchen'. "
                    "Stop Jupyter, `source ~/FYP-Radiance/figs/figs_env.sh`, relaunch.")
else:
    ok(f"conda env: {env}")

if str(PROJECT_ROOT) not in sys.executable:
    warn(f"python is {sys.executable} - not under {PROJECT_ROOT}")
else:
    ok(f"python: {sys.executable}")

for p, label in [(REPO, "repo"), (RAW_VIDEO, "raw video")]:
    (ok if p.exists() else problems.append)(f"{label}: {p}" if p.exists()
                                            else f"missing {label}: {p}")

os.chdir(REPO)
ok(f"cwd: {Path.cwd()}")

# --- imports ---
# Several packages (nerfstudio, figs, acados_template) expose no module-level __version__,
# so fall back to installed-distribution metadata before giving up.
import importlib
from importlib.metadata import version as _dist_version, PackageNotFoundError

def _version_of(mod, m):
    v = getattr(m, "__version__", None)
    if v:
        return str(v)
    for dist in (mod, mod.replace("_", "-")):
        try:
            return _dist_version(dist)
        except PackageNotFoundError:
            continue
    return "?"

for mod, pin in [("torch", "2.1.2"), ("nerfstudio", "1.1.4"), ("gsplat", None),
                 ("hloc", None), ("figs", None), ("acados_template", None),
                 ("pycolmap", None), ("cv2", None), ("open3d", None)]:
    try:
        m = importlib.import_module(mod)
        v = _version_of(mod, m)
        if pin and v == "?":
            warn(f"{mod} imported but version undeterminable (expected {pin}) - continuing")
        elif pin and not v.startswith(pin):
            problems.append(f"{mod} is {v}, expected {pin}. "
                            "A moved torch invalidates tiny-cuda-nn and gsplat.")
        else:
            ok(f"{mod} {v}")
    except Exception as e:
        problems.append(f"import {mod} failed: {e}")

# --- binaries ---
for exe in ["colmap", "ns-train", "ffmpeg", "ffprobe", "nvidia-smi"]:
    (ok if shutil.which(exe) else problems.append)(
        f"{exe}: {shutil.which(exe)}" if shutil.which(exe) else f"{exe} not on PATH")

# --- GPU ---
import torch
if not torch.cuda.is_available():
    problems.append("torch cannot see the GPU")
else:
    total = torch.cuda.get_device_properties(0).total_memory // 1024**2
    used  = _vram_now()
    ok(f"GPU: {torch.cuda.get_device_name(0)}  {total} MiB total, {used} MiB in use")
    RESULTS["gpu"] = torch.cuda.get_device_name(0)
    RESULTS["vram_total_mib"] = total
    if total < 6144:
        warn("under 6 GB - splat training will likely OOM. Simulation is still fine.")
    if used > 1000:
        warn(f"{used} MiB already in use. ns-viewer holds ~4.6 GB; close it before training "
             "or the peak measurement is meaningless and you risk an OOM.")

print()
if problems:
    for p in problems:
        print(f"  ✗ {p}")
    raise StepFailed(f"{len(problems)} preflight problem(s) - fix before continuing")
ok("preflight clean")


  Preflight
  ✔ conda env: kitchen
  ✔ python: /home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen/bin/python
  ✔ repo: /home/intellisense05/FYP-Radiance/figs/SousVide
  ✔ raw video: /home/intellisense05/FYP-Radiance/video_captures/intellisense_capture_2.mp4
  ✔ cwd: /home/intellisense05/FYP-Radiance/figs/SousVide
  ✔ torch 2.1.2
  ✔ nerfstudio 1.1.4
  ✔ gsplat 1.0.0
  ✔ hloc 1.5
  ✔ figs 0.1.0
  ✔ acados_template 0.1
  ✔ pycolmap 4.1.1
  ✔ cv2 5.0.0
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
  ✔ open3d 0.19.0
  ✔ colmap: /home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen/bin/colmap
  ✔ ns-train: /home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen/bin/ns-train
  ✔ ffmpeg: /home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen/bin/ffmpeg
  ✔ ffprobe: /home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen/bin/ffpr

## Phase 1 — Prepare the capture

Three things happen here: inspect the source video, normalise it to the validated format, and
prove the ArUco marker is actually usable *before* committing two hours of compute.

The scan is the highest-value cell in the notebook. A wrong `MARKER_ID` costs you the entire
SfM run before it surfaces as an alignment error.

In [5]:
# ── 3. Probe the raw video ───────────────────────────────────────────────────────
section("Source video")

def probe(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
         "stream=codec_name,pix_fmt,width,height,r_frame_rate,nb_frames,duration",
         "-of", "json", str(path)], capture_output=True, text=True, check=True).stdout
    s = json.loads(out)["streams"][0]
    num, den = s["r_frame_rate"].split("/")
    s["fps"] = float(num) / float(den)
    return s

src = probe(RAW_VIDEO)
size_gb = RAW_VIDEO.stat().st_size / 1024**3
info(f"codec      {src['codec_name']}  ({src['pix_fmt']})")
info(f"resolution {src['width']}x{src['height']} @ {src['fps']:.2f} fps")
info(f"duration   {float(src['duration']):.1f} s, {src.get('nb_frames','?')} frames")
info(f"size       {size_gb:.2f} GB")
RESULTS["source"] = {k: src.get(k) for k in
                     ("codec_name","pix_fmt","width","height","fps","nb_frames","duration")}

if src["pix_fmt"] not in ("yuv420p", "yuvj420p"):
    warn(f"{src['pix_fmt']} is not 8-bit 4:2:0 - likely HDR/10-bit. OpenCV may decode it "
         "slowly or with shifted colour. The transcode below fixes this.")
needs_transcode = (src["width"] != TARGET_W or src["height"] != TARGET_H
                   or src["fps"] > TARGET_FPS + 1 or src["codec_name"] != "h264")
print()
(warn if needs_transcode else ok)(
    "transcode needed" if needs_transcode else "already in target format")


  Source video
    codec      h264  (yuv420p)
    resolution 1920x1080 @ 60.00 fps
    duration   243.5 s, 14602 frames
    size       0.80 GB

  ! transcode needed


In [6]:
# ── 4. Transcode into gsplats/capture/ ───────────────────────────────────────────
# FiGS globs gsplats/capture/ by substring, so the file must live there and the name must
# be unambiguous - two files matching SCENE raises a ValueError.
section("Stage capture")

CAPTURE_MP4.parent.mkdir(parents=True, exist_ok=True)

rivals = [p for p in CAPTURE_MP4.parent.glob("*") if SCENE in p.name and p != CAPTURE_MP4]
if rivals:
    raise StepFailed(f"filename collision - these also match '{SCENE}': "
                     f"{[p.name for p in rivals]}. FiGS will raise on ambiguous glob.")

if CAPTURE_MP4.exists() and not FORCE_TRANSCODE:
    dst = probe(CAPTURE_MP4)
    ok(f"already staged: {CAPTURE_MP4.name} "
       f"({dst['width']}x{dst['height']} @ {dst['fps']:.1f}fps, "
       f"{CAPTURE_MP4.stat().st_size/1024**2:.0f} MB)")
    info("set FORCE_TRANSCODE = True to redo")
else:
    t0 = time.time()
    warn("transcoding - 4K HEVC decode is the bottleneck, expect several minutes")
    sh(["ffmpeg", "-y", "-i", str(RAW_VIDEO),
        "-vf", f"scale={TARGET_W}:{TARGET_H},format=yuv420p", "-r", str(TARGET_FPS),
        "-c:v", "libx264", "-crf", "18", "-preset", "medium", "-an", str(CAPTURE_MP4)])
    dst = probe(CAPTURE_MP4)
    ok(f"wrote {CAPTURE_MP4.name} in {elapsed(t0)} "
       f"({CAPTURE_MP4.stat().st_size/1024**2:.0f} MB)")

RESULTS["capture"] = {"path": str(CAPTURE_MP4), "width": dst["width"],
                      "height": dst["height"], "fps": dst["fps"]}


  Stage capture
  ! transcoding - 4K HEVC decode is the bottleneck, expect several minutes
ffmpeg version 8.1.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 14.3.0 (conda-forge gcc 14.3.0-19)
  configuration: --prefix=/home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1784313240697/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1784313240697/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=x86_64-conda-linux-gnu-nm --ar=x86_64-conda-linux-gnu-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --disable-gnutls --enable-libvpx --enable-libass --enable-libplacebo --enable-pthreads --enable-alsa --enable-libpulse --enable-libvpl --enable-vaapi --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3

In [7]:
# ── 5. ArUco scan ────────────────────────────────────────────────────────────────
# Proves the marker is present, identifies its ID, and - just as important - shows how the
# detections are DISTRIBUTED IN TIME. Detections clustered in one place still satisfy
# NUM_MARKED but give the RANSAC similarity fit a short baseline and a noise-sensitive result.
section("ArUco marker scan")

import cv2, numpy as np

cap = cv2.VideoCapture(str(CAPTURE_MP4))
fps = cap.get(cv2.CAP_PROP_FPS)
det = cv2.aruco.ArucoDetector(cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50),
                              cv2.aruco.DetectorParameters())
STRIDE = 2
n = 0; hits = []; all_ids = {}
t0 = time.time()
while True:
    okf, frame = cap.read()
    if not okf: break
    n += 1
    if n % STRIDE: continue
    corners, ids, _ = det.detectMarkers(frame)
    if ids is None: continue
    for k, i in enumerate(ids.flatten()):
        all_ids[int(i)] = all_ids.get(int(i), 0) + 1
        if int(i) == MARKER_ID:
            px = float(cv2.contourArea(corners[k].reshape(-1, 2).astype(np.float32))) ** 0.5
            hits.append((n / fps, px))
cap.release()

dur = n / fps
info(f"scanned {n} frames (every {STRIDE}nd) in {elapsed(t0)}")
info(f"all IDs seen: {all_ids}")
spurious = {k: v for k, v in all_ids.items() if k != MARKER_ID}
if spurious:
    info(f"ignoring {spurious} - single-digit counts are ArUco false positives")

print(f"\n  {'window':>12}  {'hits':>5}  {'median px':>9}")
windows = 0
for lo in range(0, int(dur) + 1, 10):
    w = [(t, s) for t, s in hits if lo <= t < lo + 10]
    if w: windows += 1
    print(f"  {lo:4d}-{lo+10:3d}s  {len(w):5d}  "
          f"{np.median([s for _, s in w]) if w else 0:9.0f}  {'#' * min(50, len(w))}")

est_marked = len(hits) * STRIDE
med_px = float(np.median([s for _, s in hits])) if hits else 0
print()
RESULTS["aruco"] = {"marker_id": MARKER_ID, "marker_length_m": MARKER_LENGTH,
                    "detections_scanned": len(hits), "est_marked_frames": est_marked,
                    "median_px": round(med_px, 1), "windows_10s": windows,
                    "spurious_ids": spurious}

if MARKER_ID not in all_ids:
    raise StepFailed(
        f"marker id {MARKER_ID} never detected. Either the printed marker is from a different "
        f"dictionary, or it was never clearly in shot. IDs seen: {all_ids}. "
        "No configuration change fixes this - it is a re-shoot.")
ok(f"marker {MARKER_ID} found in ~{est_marked} frames (need {NUM_MARKED})")

if est_marked < NUM_MARKED * 3:
    warn(f"thin pool ({est_marked} vs {NUM_MARKED} needed). FiGS subsamples across time, so "
         "it may not find enough. Consider lowering NUM_MARKED or re-shooting.")
if windows < 3:
    warn(f"detections span only {windows} 10s window(s). The alignment fit will have a short "
         "baseline and be noise-sensitive - scale error propagates into every waypoint.")
else:
    ok(f"spread across {windows} separate 10s windows - good alignment baseline")
if med_px < 40:
    warn(f"median apparent size {med_px:.0f} px is small; corner localisation will be "
         "imprecise. A larger printed marker helps on the next capture.")
else:
    ok(f"median apparent size {med_px:.0f} px")


  ArUco marker scan
    scanned 7308 frames (every 2nd) in 1m 7s
    all IDs seen: {0: 1563, 17: 3, 37: 2}
    ignoring {17: 3, 37: 2} - single-digit counts are ArUco false positives

        window   hits  median px
     0- 10s    149        164  ##################################################
    10- 20s    150        224  ##################################################
    20- 30s    150        161  ##################################################
    30- 40s     27        148  ###########################
    40- 50s      0          0  
    50- 60s      0          0  
    60- 70s      2         71  ##
    70- 80s      0          0  
    80- 90s    147        131  ##################################################
    90-100s    149        172  ##################################################
   100-110s    150        160  ##################################################
   110-120s      1        181  #
   120-130s      0          0  
   130-140s    113         68  #####

## Phase 2 — Configuration and compatibility

The capture config carries the two numbers that define metric scale. The compatibility cell
handles a real, reproducible break: `install_figs.sh` leaves `pycolmap` unpinned, so a rebuild
picks up whatever is newest, and newer pycolmap renamed an `import_images` keyword that the
vendored hloc still uses.

In [8]:
# ── 6. Capture config ────────────────────────────────────────────────────────────
section("Capture config")

cfg_path = REPO / "configs" / "captures" / f"{SCENE}.json"
cfg = {"camera": None,
       "extractor": {"num_images": NUM_IMAGES, "num_marked": NUM_MARKED,
                     "marker_length": MARKER_LENGTH, "marker_id": MARKER_ID}}
cfg_path.write_text(json.dumps(cfg, indent=4) + "\n")
print(textwrap.indent(cfg_path.read_text(), "    "))
ok(f"wrote {cfg_path.relative_to(REPO)}")
warn(f"marker_length = {MARKER_LENGTH} m must be the BLACK SQUARE's side, not the paper or "
     "the white quiet zone. Nothing downstream can detect an error here - a wrong value "
     "produces a perfectly reconstructed splat at the wrong scale.")
RESULTS["capture_cfg"] = cfg["extractor"]


  Capture config
    {
        "camera": null,
        "extractor": {
            "num_images": 300,
            "num_marked": 40,
            "marker_length": 0.18,
            "marker_id": 0
        }
    }

  ✔ wrote configs/captures/intellisense_02.json
  ! marker_length = 0.18 m must be the BLACK SQUARE's side, not the paper or the white quiet zone. Nothing downstream can detect an error here - a wrong value produces a perfectly reconstructed splat at the wrong scale.


In [9]:
# ── 7. hloc / pycolmap compatibility ─────────────────────────────────────────────
# Detects the API the installed pycolmap exposes and patches the vendored hloc to match.
# Idempotent: safe to re-run. Keeps a .bak, because this is a git submodule and
# `install_figs.sh --redo clone` silently reverts it.
section("hloc / pycolmap compatibility")

import pycolmap, inspect
recon_py = REPO / "FiGS" / "Hierarchical-Localization" / "hloc" / "reconstruction.py"
info(f"pycolmap {pycolmap.__version__}")

doc = pycolmap.import_images.__doc__ or ""
wants_image_names = "image_names" in doc
recon_src = recon_py.read_text()
uses_image_list = "image_list=image_list" in recon_src

if wants_image_names and uses_image_list:
    bak = recon_py.with_suffix(".py.bak")
    if not bak.exists():
        shutil.copy(recon_py, bak); info(f"backup -> {bak.name}")
    patched = recon_src.replace("image_list=image_list", "image_names=image_list")
    recon_py.write_text(patched)
    ok("patched hloc: import_images(image_list=...) -> image_names=...")
elif wants_image_names:
    ok("hloc already patched for the image_names API")
else:
    ok("pycolmap uses the image_list API - no patch needed")

# The other three call sites pass dict options, which pybind converts to whatever options
# object the installed pycolmap expects. Verified compatible on pycolmap 4.1.1.
for fn in ("incremental_mapping", "verify_matches"):
    (ok if hasattr(pycolmap, fn) else warn)(f"pycolmap.{fn} present")
RESULTS["pycolmap_version"] = pycolmap.__version__


  hloc / pycolmap compatibility
    pycolmap 4.1.1
  ✔ hloc already patched for the image_names API
  ✔ pycolmap.incremental_mapping present
  ✔ pycolmap.verify_matches present


## Phase 3 — Generate the splat

The long one. Frame extraction, then hloc feature extraction and matching, then COLMAP
incremental mapping, then ArUco metric alignment, then `ns-train splatfacto` to 30,000 steps.

Runs as a subprocess so a crash cannot poison the kernel, with output streamed live and VRAM
sampled in the background. **Reference run: 1h 34m total** — SuperGlue matching 38 min, COLMAP
mapping 5.7 min, training 34 min, peak 5203 MiB.

hloc caches features and matches to h5, so a failure after matching resumes cheaply.

In [11]:
# ── 8. generate_gsplat ───────────────────────────────────────────────────────────
section("Generate Gaussian Splat")

ckpts = sorted((REPO / "gsplats" / "workspace" / "outputs" / SCENE).rglob("*.ckpt")) \
        if (REPO / "gsplats" / "workspace" / "outputs" / SCENE).exists() else []

if ckpts and not FORCE_GSPLAT:
    ok(f"checkpoint already exists: {ckpts[-1].relative_to(REPO)}")
    info(f"{ckpts[-1].stat().st_size/1024**2:.0f} MB - set FORCE_GSPLAT = True to retrain")
else:
    if _vram_now() > 1000:
        warn(f"{_vram_now()} MiB already allocated - close ns-viewer first, it holds ~4.6 GB")
    script = (f"import figs.render.capture_generation as pg\n"
              f"pg.generate_gsplat({SCENE!r}, capture_cfg_name={SCENE!r})\n")
    log = HOME / f"gsplat_{SCENE}_{datetime.now():%Y-%m-%d_%H%M}.log"
    info(f"logging to {log}")
    t0 = time.time()
    with VramMonitor() as vm:
        rc, out = sh([sys.executable, "-c", script], cwd=REPO, check=False)
    log.write_text(out)
    if rc != 0:
        raise StepFailed(diagnose(out) or f"generate_gsplat failed (rc={rc}); see {log}")
    RESULTS["train_peak_vram_mib"] = vm.peak
    RESULTS["gsplat_wallclock"] = elapsed(t0)
    ok(f"completed in {elapsed(t0)}, peak VRAM {vm.peak} MiB")


  Generate Gaussian Splat
    logging to /home/intellisense05/gsplat_intellisense_02_2026-08-07_0914.log
( ●    ) Copying images...
(  ●   ) Copying images...
(   ●  ) Copying images...
(    ● ) Copying images...
(     ●) Copying images...
(    ● ) Copying images...
(   ●  ) Copying images...
(  ●   ) Copying images...
( ●    ) Copying images...
(●     ) Copying images...
( ●    ) Copying images...
(  ●   ) Copying images...
(   ●  ) Copying images...
(    ● ) Copying images...
(     ●) Copying images...
(    ● ) Copying images...
(   ●  ) Copying images...
(  ●   ) Copying images...
( ●    ) Copying images...
(●     ) Copying images...
( ●    ) Copying images...
(  ●   ) Copying images...
(   ●  ) Copying images...
(    ● ) Copying images...
(     ●) Copying images...
(    ● ) Copying images...
(   ●  ) Copying images...
(  ●   ) Copying images...
( ●    ) Copying images...
(●     ) Copying images...
( ●    ) Copying images...
(  ●   ) Copying images...
(   ●  ) Copying images...
(  

StepFailed: pycolmap/hloc API drift. The vendored hloc calls import_images(image_list=...) but newer pycolmap renamed it to image_names. Re-run the hloc compatibility cell (Phase 2).

--- matched in output ---
, image_dir, image_list=references, feature_path=features)  # type: ignore
  File "/home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 115, in decorate_context
    return func(*args, **kwargs)
  File "/home/intellisense05/FYP-Radiance/figs/SousVide/FiGS/Hierarchical-Localization/hloc/extract_features.py", line 265, in main
    for idx, data in enumerate(tqdm(loader)):
  File "/home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen/lib/python3.10/site-packages/tqdm/std.py", line 1187, in __iter__
    for obj in iterable:
  File "/home/intellisense05/FYP-Radiance/figs/miniconda3/envs/kitchen/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 630, in __next__
    data = self._next_data()

In [12]:
# ── 9. Verify the reconstruction ─────────────────────────────────────────────────
# generate_gsplat does not surface COLMAP's registration statistics and does not fail on a
# partial reconstruction - it silently trains a worse splat. This is the check that catches it.
section("Verify reconstruction")

import open3d as o3d

n_images = len(list((WORKSPACE / "images").glob("*")))
tf = json.loads((WORKSPACE / "transforms.json").read_text())
n_reg = len(tf["frames"])
pcd = o3d.io.read_point_cloud(str(WORKSPACE / "sparse_pc.ply"))
n_pts = len(pcd.points)
ckpt = sorted((REPO / "gsplats" / "workspace" / "outputs" / SCENE).rglob("*.ckpt"))[-1]
ckpt_mb = ckpt.stat().st_size / 1024**2
pct = 100 * n_reg / max(n_images, 1)

info(f"extracted images   {n_images}")
info(f"registered poses   {n_reg}  ({pct:.1f}%)")
info(f"sparse points      {n_pts}")
info(f"checkpoint         {ckpt_mb:.0f} MB  {ckpt.relative_to(REPO)}")
RESULTS["sfm"] = {"images": n_images, "registered": n_reg, "pct": round(pct, 1),
                  "sparse_points": n_pts, "ckpt_mb": round(ckpt_mb),
                  "ckpt": str(ckpt.relative_to(REPO))}

print()
if pct < 90:
    raise StepFailed(
        f"only {pct:.1f}% of images registered. Below ~90% means insufficient overlap or "
        "motion blur during filming. Everything downstream inherits the holes - re-shoot "
        "rather than proceeding. Walk slower, keep generous overlap between viewpoints.")
ok(f"{pct:.1f}% registered")
if n_pts < 5000:
    warn(f"only {n_pts} sparse points - thin reconstruction, expect a weak splat")
else:
    ok(f"{n_pts} sparse points")
ok(f"checkpoint present ({ckpt_mb:.0f} MB)")


  Verify reconstruction
    extracted images   300
    registered poses   300  (100.0%)
    sparse points      43623
    checkpoint         529 MB  gsplats/workspace/outputs/intellisense_02/splatfacto/2026-08-07_103309/nerfstudio_models/step-000029999.ckpt

  ✔ 100.0% registered
  ✔ 43623 sparse points
  ✔ checkpoint present (529 MB)


## Phase 4 — Frame conversion and course validation

**The sign convention is the trap in this whole pipeline.** `transforms.json` is z-up
(camera heights are positive). Course files are z-down — `circuit.json` writes 0.7 m altitude
as `z = -0.70`. The mapping is a 180° rotation about x:

```
course = (x, -y, -z)   relative to transforms.json
```

Derived empirically from the known-good `backroom` + `circuit` pair, and re-verified here each
run. Get it wrong and the drone flies below the floor: the sim runs without error and renders
mush, which reads as "the splat is bad."

In [13]:
# ── 10. Capture bounds → usable course box ──────────────────────────────────────
section("Capture bounds")

# Duration is only used for the walking-speed figure. Take it from RESULTS, fall back to
# probing the staged file, and degrade gracefully rather than failing the whole cell.
def _video_duration():
    d = RESULTS.get("source", {}).get("duration")
    if d:
        return float(d)
    try:
        return float(probe(CAPTURE_MP4)["duration"])
    except Exception:
        return None

VID_DUR = _video_duration()
P = np.array([f["transform_matrix"] for f in tf["frames"]])[:, :3, 3]
path_len = float(np.linalg.norm(np.diff(P, axis=0), axis=1).sum())

print("  splat frame (z-up, metres):")
for i, ax in enumerate("xyz"):
    print(f"    {ax}: {P[:,i].min():+6.2f} .. {P[:,i].max():+6.2f}   "
          f"(span {P[:,i].max()-P[:,i].min():5.2f}, median {np.median(P[:,i]):+6.2f})")
if VID_DUR:
    info(f"camera path length {path_len:.1f} m over {VID_DUR:.0f} s "
         f"→ {path_len/VID_DUR:.2f} m/s")
else:
    info(f"camera path length {path_len:.1f} m (duration unknown - re-run the probe cell)")

def splat_to_course(p):  return np.array([ p[0], -p[1], -p[2]])
def course_to_splat(p):  return np.array([ p[0], -p[1], -p[2]])   # self-inverse

lo, hi = P.min(axis=0), P.max(axis=0)
c_lo = np.minimum(splat_to_course(lo), splat_to_course(hi))
c_hi = np.maximum(splat_to_course(lo), splat_to_course(hi))
print("\n  course frame (z-down) - full extent of camera travel:")
for i, ax in enumerate("xyz"):
    print(f"    {ax}: {c_lo[i]:+6.2f} .. {c_hi[i]:+6.2f}")

MARGIN = 0.5
print(f"\n  recommended waypoint box (inset {MARGIN} m):")
for i, ax in enumerate("xyz"):
    print(f"    {ax}: {c_lo[i]+MARGIN:+6.2f} .. {c_hi[i]-MARGIN:+6.2f}")

RESULTS["bounds_splat"] = {ax: [round(float(lo[i]),2), round(float(hi[i]),2)]
                           for i, ax in enumerate("xyz")}
RESULTS["path_length_m"] = round(path_len, 1)

if VID_DUR and path_len / VID_DUR > 1.5:
    warn("camera moved faster than ~1.5 m/s. It worked here, but slower walking is the first "
         "thing to change if a future capture reconstructs poorly.")
warn("These are bounds of where the CAMERA went, not a map of free space. Check the course "
     "against the splat in ns-viewer before flying - furniture does not appear in a bbox.")


  Capture bounds
  splat frame (z-up, metres):
    x:  -1.27 ..  +1.81   (span  3.09, median  +0.06)
    y:  -3.33 ..  +1.39   (span  4.71, median  -0.87)
    z:  +0.50 ..  +1.65   (span  1.14, median  +1.15)
    camera path length 247.2 m over 244 s → 1.01 m/s

  course frame (z-down) - full extent of camera travel:
    x:  -1.27 ..  +1.81
    y:  -1.39 ..  +3.33
    z:  -1.65 ..  -0.50

  recommended waypoint box (inset 0.5 m):
    x:  -0.77 ..  +1.31
    y:  -0.89 ..  +2.83
    z:  -1.15 ..  -1.00
  ! These are bounds of where the CAMERA went, not a map of free space. Check the course against the splat in ns-viewer before flying - furniture does not appear in a bbox.


In [14]:
# ── 11. Validate the course against the capture ──────────────────────────────────
# Catches the failure that produces no error at all: waypoints outside the filmed volume,
# which render as blurry floaters exactly at the offending points.
section("Course validation")

course_path = REPO / "configs" / "courses" / f"{COURSE}.json"
if not course_path.exists():
    raise StepFailed(f"{course_path.relative_to(REPO)} not found. Author it using the "
                     "recommended box above, or copy an existing course as a starting point.")

kf = json.loads(course_path.read_text())["waypoints"]["keyframes"]
rows, outside = [], 0
for name, k in kf.items():
    fo = k["fo"]
    p_course = np.array([(f[0] if isinstance(f, list) else f) for f in fo[:3]], dtype=float)
    p_splat  = course_to_splat(p_course)
    inside   = bool(np.all(p_splat >= lo) and np.all(p_splat <= hi))
    outside += (not inside)
    rows.append((name, k["t"], p_course, p_splat, inside))

print(f"  {'keyframe':<8} {'t':>6}  {'course (x,y,z)':<22} {'splat (x,y,z)':<22} in?")
for name, t, pc, ps, inside in rows:
    print(f"  {name:<8} {t:6.2f}  "
          f"({pc[0]:+5.2f},{pc[1]:+5.2f},{pc[2]:+5.2f})      "
          f"({ps[0]:+5.2f},{ps[1]:+5.2f},{ps[2]:+5.2f})      "
          f"{'✔' if inside else '✗ OUTSIDE'}")

print()
RESULTS["course"] = {"name": COURSE, "keyframes": len(kf), "outside": outside}
if outside:
    warn(f"{outside} waypoint(s) outside the captured volume. The simulation will run without "
         "error and render floaters at exactly those points. Shrink the course toward the "
         "recommended box above.")
else:
    ok(f"all {len(kf)} keyframes inside the captured volume")


  Course validation
  keyframe      t  course (x,y,z)         splat (x,y,z)          in?
  k0         0.00  (+0.40,+0.20,-1.10)      (+0.40,-0.20,+1.10)      ✔
  k1         3.60  (+0.40,-1.60,-1.10)      (+0.40,+1.60,+1.10)      ✗ OUTSIDE
  k2         4.40  (+0.20,-1.80,-1.10)      (+0.20,+1.80,+1.10)      ✗ OUTSIDE
  k3         7.60  (-1.40,-1.80,-1.10)      (-1.40,+1.80,+1.10)      ✗ OUTSIDE
  k4         8.40  (-1.60,-1.60,-1.10)      (-1.60,+1.60,+1.10)      ✗ OUTSIDE
  k5        11.60  (-1.60,+0.00,-1.10)      (-1.60,-0.00,+1.10)      ✗ OUTSIDE
  k6        12.40  (-1.40,+0.20,-1.10)      (-1.40,-0.20,+1.10)      ✗ OUTSIDE
  k7        16.00  (+0.40,+0.20,-1.10)      (+0.40,-0.20,+1.10)      ✔

  ! 6 waypoint(s) outside the captured volume. The simulation will run without error and render floaters at exactly those points. Shrink the course toward the recommended box above.


## Phase 5 — Simulate

Loads the splat, builds the ACADOS MPC around a minimum-snap trajectory through the course, and
runs the closed loop: MPC solves rates → 9-DOF dynamics integrates → the splat renders the
onboard camera at every control step.

Runs in-process so `Rgb`, `Xro` and friends stay available for analysis afterwards. That does
mean the kernel holds the splat on the GPU — restart the kernel to release it.

Expect acados C codegen on the first run for a new course: slow once, cached after.

In [17]:
# ── 12. Fly it ───────────────────────────────────────────────────────────────────
section("Simulate")

if _vram_now() > 1000:
    warn(f"{_vram_now()} MiB in use before loading - close ns-viewer for a clean measurement")

from figs.simulator import Simulator
from figs.control.vehicle_rate_mpc import VehicleRateMPC
import figs.visualize.generate_videos as gv

t0 = time.time()
with VramMonitor(interval=2) as vm:
    sim = Simulator(SCENE, METHOD, FRAME)
    ctl = VehicleRateMPC(PILOT, COURSE, FRAME)
    tA, tB, x0 = ctl.tXUd[0, 0], ctl.tXUd[-1, 0], ctl.tXUd[0, 1:11]
    info(f"trajectory {tA:.2f}s → {tB:.2f}s at {ctl.hz} Hz")
    Tro, Xro, Uro, Fro, Rgb, Dpt, Aux = sim.simulate(ctl, tA, tB, x0)
    gv.images_to_mp4(Rgb, OUT_MP4, ctl.hz)

wall = elapsed(t0)
out = REPO / OUT_MP4
ok(f"{OUT_MP4}  {out.stat().st_size/1024**2:.1f} MB  ({Rgb.shape[0]} frames) in {wall}")
RESULTS["sim"] = {"video": OUT_MP4, "frames": int(Rgb.shape[0]), "hz": float(ctl.hz),
                  "wallclock": wall, "peak_vram_mib": vm.peak,
                  "duration_s": round(float(tB - tA), 2)}

# Tracking error against the reference trajectory - a cheap quality signal on the MPC.
try:
    ref = np.array([np.interp(Tro, ctl.tXUd[:, 0], ctl.tXUd[:, i]) for i in (1, 2, 3)]).T
    err = np.linalg.norm(Xro[:, :3] - ref, axis=1)
    info(f"tracking error: mean {err.mean():.3f} m, max {err.max():.3f} m")
    RESULTS["sim"]["track_err_mean_m"] = round(float(err.mean()), 3)
    RESULTS["sim"]["track_err_max_m"]  = round(float(err.max()), 3)
    if err.max() > 0.5:
        warn("large tracking error - the trajectory may be too aggressive for the MPC. "
             "Stretch the course `t` values and retry.")
except Exception as e:
    info(f"(tracking error not computed: {e})")


  Simulate


Loading latest checkpoint from load_dir

✅ Done loading checkpoint from 
outputs/intellisense_02/splatfacto/2026-08-07_103309/nerfstudio_models/step-000029999.ckpt

Please export ACADOS_SOURCE_DIR to avoid this warning.
 If there is an incompatibility with the CasADi generated code, please consider changing your CasADi version.
Version 3.7.2 currently in use.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.

  VRAM baseline 792 MiB → peak 792 MiB


OSError: libqpOASES_e.so: cannot open shared object file: No such file or directory

In [ ]:
# ── 13. Run record ───────────────────────────────────────────────────────────────
section("Summary")

RESULTS["finished_at"] = datetime.now().isoformat(timespec="seconds")
runs = REPO / "runs"; runs.mkdir(exist_ok=True)
rec = runs / f"{SCENE}_{datetime.now():%Y-%m-%d_%H%M}.json"
rec.write_text(json.dumps(RESULTS, indent=2, default=str) + "\n")

# resolve the nerfstudio config next to the checkpoint, for the ns-viewer hint below
_ck  = Path(RESULTS.get("sfm", {}).get("ckpt", ""))
_cfg = (_ck.parent.parent / "config.yml") if _ck.name.endswith(".ckpt") else None
_cfg_rel = _cfg.relative_to("gsplats/workspace") if _cfg else "<config.yml>"

print(json.dumps(RESULTS, indent=2, default=str))
print()
ok(f"run record: {rec.relative_to(REPO)}")
print(f"""
  Next:
    - watch the flight:   {REPO / OUT_MP4}
    - inspect the splat:  cd {REPO}/gsplats/workspace
                          ns-viewer --load-config {_cfg_rel}
                          then browse http://localhost:7007  (holds ~4.6 GB VRAM)
    - pull to Windows:    scp intellisense05@10.8.100.30:{REPO / OUT_MP4} .

  If the video shows floaters at specific spots, those waypoints reached past your capture
  coverage - shrink the course. Uniform blur everywhere is a different problem: revisit the
  reconstruction statistics in cell 9.
""")